## BNPL Credit Risk Modelling

This notebook prepares the point-in-time BNPL modelling dataset and develops a time-based default prediction framework using the primary 90-day default outcome.

The modelling framework compares two information structures:

1. Full Information Model, using transaction, behavioural and available credit-risk variables.
2. Behavioural Model, excluding conventional credit-score information to assess how much predictive value is contained in BNPL transaction behaviour alone.

Logistic Regression is used as an interpretable baseline, while Random Forest is used as a nonlinear benchmark.

The modelling process follows a temporal train-validation-test design. Transactions from 2022 are used for training, 2023 for validation and model selection, and 2024 as an out-of-time test period.

All historical behavioural variables are required to be point-in-time safe, using information available before the prediction transaction. The target variable is `default_90d`.

In [0]:
# ============================================================
# LOAD PERSISTED BNPL GOLD DATA
# ============================================================

from pyspark.sql import functions as F

gold_path = "/Volumes/workspace/default/bnpl_raw/gold_bnpl"

gold_df = (
    spark.read
    .format("delta")
    .load(gold_path)
)

print("Gold rows:", gold_df.count())
print("Gold columns:", len(gold_df.columns))

Gold rows: 2000000
Gold columns: 29


In [0]:
# ============================================================
# VERIFY GOLD DATASET STRUCTURE
# ============================================================

required_gold_columns = [
    "transaction_id",
    "purchase_date",
    "customer_id",
    "merchant_category",
    "merchant_name",
    "customer_state",
    "principal_ngn",
    "interest_rate_monthly",
    "tenor_days",
    "num_installments",
    "provider",
    "credit_score",
    "first_time_customer",
    "first_payment_due",
    "default_30d",
    "default_90d",
    "prior_transaction_count",
    "prior_total_exposure",
    "prior_matured_transaction_count",
    "prior_default_count",
    "prior_default_rate",
    "previous_purchase_date",
    "days_since_previous_transaction",
    "transactions_last_30d",
    "exposure_last_30d",
    "transactions_last_60d",
    "exposure_last_60d",
    "transactions_last_90d",
    "exposure_last_90d"
]

missing_gold_columns = [
    column
    for column in required_gold_columns
    if column not in gold_df.columns
]

unexpected_gold_columns = [
    column
    for column in gold_df.columns
    if column not in required_gold_columns
]

print("Expected Gold columns:", len(required_gold_columns))
print("Actual Gold columns:", len(gold_df.columns))
print("Missing required columns:", missing_gold_columns)
print("Unexpected columns:", unexpected_gold_columns)

assert len(missing_gold_columns) == 0, (
    f"Gold dataset is missing required columns: {missing_gold_columns}"
)

assert len(gold_df.columns) == 29, (
    f"Expected 29 Gold columns, found {len(gold_df.columns)}"
)

print("\nGold structure validation passed.")

Expected Gold columns: 29
Actual Gold columns: 29
Missing required columns: []
Unexpected columns: []

Gold structure validation passed.


In [0]:
# ============================================================
# GOLD SCHEMA CHECK
# ============================================================

gold_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- purchase_date: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- principal_ngn: double (nullable = true)
 |-- interest_rate_monthly: double (nullable = true)
 |-- tenor_days: integer (nullable = true)
 |-- num_installments: integer (nullable = true)
 |-- provider: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- first_time_customer: boolean (nullable = true)
 |-- first_payment_due: timestamp (nullable = true)
 |-- default_30d: boolean (nullable = true)
 |-- default_90d: boolean (nullable = true)
 |-- prior_transaction_count: long (nullable = true)
 |-- prior_total_exposure: double (nullable = true)
 |-- previous_purchase_date: timestamp (nullable = true)
 |-- days_since_previous_transaction: integer (nullable = true)
 |-- prior_matured_transa

The modelling feature design follows the leakage controls established during data engineering. Current transaction variables are available at the lending decision, while behavioural variables are derived from information preceding the current transaction.

The three rolling default-count variables from the earlier Gold design are intentionally absent. They were removed during the corrected engineering stage because outcome-based rolling variables could introduce leakage or rely on information that would not necessarily be known at the prediction point.

Customer identifiers, transaction identifiers, dates used only for temporal ordering, and outcome variables are excluded from the predictive feature set.

In [0]:
# ============================================================
# DEFINE LEAKAGE REGISTER
# ============================================================

target_columns = [
    "default_30d",
    "default_90d"
]

identifier_columns = [
    "transaction_id",
    "customer_id"
]

temporal_columns = [
    "purchase_date"
]

excluded_columns = [
    "transaction_id",
    "customer_id",
    "purchase_date",
    "merchant_name",
    "previous_purchase_date",
    "first_payment_due",
    "default_30d",
    "default_90d"
]

forbidden_feature_columns = [
    "default_30d",
    "default_90d",
    "defaults_last_30d",
    "defaults_last_60d",
    "defaults_last_90d"
]

print("Primary target:", "default_90d")
print("Secondary target:", "default_30d")
print("\nExcluded identifiers:", identifier_columns)
print("Excluded temporal fields:", temporal_columns)
print("Forbidden target-derived features:", forbidden_feature_columns)

assert "default_90d" in target_columns
assert "default_90d" in forbidden_feature_columns

Primary target: default_90d
Secondary target: default_30d

Excluded identifiers: ['transaction_id', 'customer_id']
Excluded temporal fields: ['purchase_date']
Forbidden target-derived features: ['default_30d', 'default_90d', 'defaults_last_30d', 'defaults_last_60d', 'defaults_last_90d']


In [0]:
# ============================================================
# PRIMARY TARGET DISTRIBUTION
# ============================================================

target_distribution = (
    gold_df
    .groupBy("default_90d")
    .count()
    .withColumn(
        "percentage",
        F.round(
            F.col("count") / F.lit(gold_df.count()) * 100,
            4
        )
    )
    .orderBy("default_90d")
)

display(target_distribution)

default_90d,count,percentage
false,1839999,92.0
true,160001,8.0001


The primary modelling outcome is `default_90d`, representing whether the transaction reaches a 90-day default outcome. The target distribution contains 1,839,999 non-default transactions (92.0%) and 160,001 default transactions (8.0%). This indicates a moderately imbalanced classification problem, making accuracy alone insufficient for evaluating model performance. ROC-AUC, PR-AUC, precision, recall, specificity and F1 are therefore considered together when assessing the models.

In [0]:
# ============================================================
# DEFAULT RATE OVER TIME
# Diagnostic only, not a new prediction target
# ============================================================

monthly_target = (
    gold_df
    .withColumn(
        "year_month",
        F.date_format("purchase_date", "yyyy-MM")
    )
    .groupBy("year_month")
    .agg(
        F.count("*").alias("transactions"),
        F.sum(
            F.when(F.col("default_90d") == True, 1).otherwise(0)
        ).alias("defaults")
    )
    .withColumn(
        "default_rate",
        F.round(
            F.col("defaults") / F.col("transactions") * 100,
            3
        )
    )
    .orderBy("year_month")
)

display(monthly_target)

year_month,transactions,defaults,default_rate
2022-01,57038,4604,8.072
2022-02,51261,4201,8.195
2022-03,56494,4573,8.095
2022-04,54586,4289,7.857
2022-05,56330,4524,8.031
2022-06,54673,4429,8.101
2022-07,56435,4637,8.217
2022-08,56969,4542,7.973
2022-09,55002,4391,7.983
2022-10,57063,4547,7.968


Monthly 90-day default rates remain broadly stable around the overall 8% level across the 2022 to 2024 observation period. No pronounced temporal deterioration or improvement in target prevalence is visible. This stability supports the use of chronological validation and an out-of-time test period. Because the dataset is synthetic, this pattern should be interpreted as a characteristic of the constructed dataset rather than evidence of actual Nigerian BNPL market stability.

In [0]:
# ============================================================
# CREATE CHRONOLOGICAL TRAIN, VALIDATION AND OUT-OF-TIME TEST DATASETS
# ============================================================

train_df = gold_df.filter(
    (F.col("purchase_date") >= F.lit("2022-01-01")) &
    (F.col("purchase_date") < F.lit("2023-01-01"))
)

validation_df = gold_df.filter(
    (F.col("purchase_date") >= F.lit("2023-01-01")) &
    (F.col("purchase_date") < F.lit("2024-01-01"))
)

test_df = gold_df.filter(
    (F.col("purchase_date") >= F.lit("2024-01-01")) &
    (F.col("purchase_date") < F.lit("2025-01-01"))
)

print("Train rows:", train_df.count())
print("Validation rows:", validation_df.count())
print("Test rows:", test_df.count())

Train rows: 666657
Validation rows: 667097
Test rows: 666246


In [0]:
# ============================================================
# VERIFY TEMPORAL SPLIT
# ============================================================

split_check = [
    ("Train", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]

for name, df in split_check:

    row = (
        df
        .select(
            F.min("purchase_date").alias("min_date"),
            F.max("purchase_date").alias("max_date"),
            F.count("*").alias("rows"),
            F.avg(
                F.when(
                    F.col("default_90d") == True,
                    1.0
                ).otherwise(0.0)
            ).alias("default_rate")
        )
        .collect()[0]
    )

    print(
        f"{name}: "
        f"{row['min_date']} → {row['max_date']} | "
        f"Rows = {row['rows']:,} | "
        f"Default rate = {row['default_rate'] * 100:.3f}%"
    )

Train: 2022-01-01 00:00:00 → 2022-12-31 00:00:00 | Rows = 666,657 | Default rate = 8.040%
Validation: 2023-01-01 00:00:00 → 2023-12-31 00:00:00 | Rows = 667,097 | Default rate = 7.958%
Test: 2024-01-01 00:00:00 → 2024-12-30 00:00:00 | Rows = 666,246 | Default rate = 8.002%


The modelling data are separated chronologically into 2022 for training, 2023 for validation and 2024 for the final out-of-time test. The resulting datasets contain 666,657 training transactions, 667,097 validation transactions and 666,246 test transactions. The chronological separation prevents future transactions from being used to train or select the model used for earlier periods and provides a realistic temporal evaluation structure.

In [0]:
# ============================================================
# VERIFY TEMPORAL SEPARATION
# ============================================================

train_max_date = train_df.select(
    F.max("purchase_date").alias("max_date")
).collect()[0]["max_date"]

validation_min_date = validation_df.select(
    F.min("purchase_date").alias("min_date")
).collect()[0]["min_date"]

validation_max_date = validation_df.select(
    F.max("purchase_date").alias("max_date")
).collect()[0]["max_date"]

test_min_date = test_df.select(
    F.min("purchase_date").alias("min_date")
).collect()[0]["min_date"]

assert train_max_date < validation_min_date
assert validation_max_date < test_min_date

print("Temporal separation validation passed.")
print("Training precedes validation.")
print("Validation precedes out-of-time test.")

Temporal separation validation passed.
Training precedes validation.
Validation precedes out-of-time test.


In [0]:
# ============================================================
# CUSTOMER OVERLAP ACROSS TEMPORAL SPLITS
# ============================================================

train_customers = train_df.select("customer_id").distinct()
validation_customers = validation_df.select("customer_id").distinct()
test_customers = test_df.select("customer_id").distinct()

train_validation_overlap = (
    train_customers
    .join(validation_customers, "customer_id", "inner")
    .count()
)

validation_test_overlap = (
    validation_customers
    .join(test_customers, "customer_id", "inner")
    .count()
)

train_test_overlap = (
    train_customers
    .join(test_customers, "customer_id", "inner")
    .count()
)

print(
    "Train ↔ Validation customer overlap:",
    train_validation_overlap
)

print(
    "Validation ↔ Test customer overlap:",
    validation_test_overlap
)

print(
    "Train ↔ Test customer overlap:",
    train_test_overlap
)

Train ↔ Validation customer overlap: 266898
Validation ↔ Test customer overlap: 266613
Train ↔ Test customer overlap: 266168


Customers may legitimately appear in multiple temporal periods because BNPL borrowers can originate multiple transactions over time. Customer overlap therefore does not by itself constitute target leakage in this transaction-level prediction setting. Leakage control is instead enforced through point-in-time feature construction, where historical behavioural variables are based only on information available before the prediction transaction.

In [0]:
# ============================================================
# CREATE MODELLING-SAFE HISTORY VARIABLES
# ============================================================

def prepare_model_data(df):

    return (
        df
        .withColumn(
            "has_prior_history",
            F.when(
                F.col("prior_transaction_count") > 0,
                1
            ).otherwise(0)
        )
        .withColumn(
            "days_since_previous_transaction_model",
            F.when(
                F.col("days_since_previous_transaction").isNull() |
                (F.col("days_since_previous_transaction") < 0),
                0
            ).otherwise(
                F.col("days_since_previous_transaction")
            )
        )
    )


train_model_df = prepare_model_data(train_df)
validation_model_df = prepare_model_data(validation_df)
test_model_df = prepare_model_data(test_df)

print("Train:", train_model_df.count())
print("Validation:", validation_model_df.count())
print("Test:", test_model_df.count())

Train: 666657
Validation: 667097
Test: 666246


In [0]:
# ============================================================
# POINT-IN-TIME FEATURE SANITY CHECK
# ============================================================

pit_check = (
    test_model_df
    .select(
        F.sum(
            F.when(
                F.col("previous_purchase_date") > F.col("purchase_date"),
                1
            ).otherwise(0)
        ).alias("invalid_previous_dates"),

        F.sum(
            F.when(
                F.col("days_since_previous_transaction") < 0,
                1
            ).otherwise(0)
        ).alias("negative_recency"),

        F.sum(
            F.when(
                F.col("prior_transaction_count") < 0,
                1
            ).otherwise(0)
        ).alias("negative_prior_transaction_count"),

        F.sum(
            F.when(
                F.col("prior_total_exposure") < 0,
                1
            ).otherwise(0)
        ).alias("negative_prior_exposure"),

        F.sum(
            F.when(
                F.col("prior_default_count") < 0,
                1
            ).otherwise(0)
        ).alias("negative_prior_default_count"),

        F.sum(
            F.when(
                F.col("transactions_last_30d") < 0,
                1
            ).otherwise(0)
        ).alias("negative_30d_transactions"),

        F.sum(
            F.when(
                F.col("transactions_last_60d") < 0,
                1
            ).otherwise(0)
        ).alias("negative_60d_transactions"),

        F.sum(
            F.when(
                F.col("transactions_last_90d") < 0,
                1
            ).otherwise(0)
        ).alias("negative_90d_transactions")
    )
)

display(pit_check)

invalid_previous_dates,negative_recency,negative_prior_transaction_count,negative_prior_exposure,negative_prior_default_count,negative_30d_transactions,negative_60d_transactions,negative_90d_transactions
0,0,0,0,0,0,0,0


The point-in-time validation confirms that the behavioural features do not contain future-looking transaction information. No negative historical counts, negative exposure values or negative recency values are detected, and no transaction has a previous transaction dated after its prediction date. Same-day transactions are permitted because transactions occurring on the same date are deterministically ordered during feature engineering using the transaction identifier as a tie-breaker. Therefore, equality between `previous_purchase_date` and `purchase_date` does not represent leakage in this dataset.

In [0]:
# ============================================================
# DEFINE MODEL FEATURE SPECIFICATIONS
# ============================================================

transaction_features = [
    "principal_ngn",
    "interest_rate_monthly",
    "tenor_days",
    "num_installments",
    "merchant_category",
    "provider",
    "customer_state"
]

behavioral_features = [
    "prior_transaction_count",
    "prior_total_exposure",
    "prior_matured_transaction_count",
    "prior_default_count",
    "prior_default_rate",
    "days_since_previous_transaction_model",
    "has_prior_history",
    "transactions_last_30d",
    "exposure_last_30d",
    "transactions_last_60d",
    "exposure_last_60d",
    "transactions_last_90d",
    "exposure_last_90d"
]

full_features = (
    transaction_features
    + [
        "credit_score",
        "first_time_customer"
    ]
    + behavioral_features
)

behavioral_model_features = (
    transaction_features
    + behavioral_features
)

print("Transaction features:", len(transaction_features))
print("Behavioural history features:", len(behavioral_features))
print("Full Information features:", len(full_features))
print(
    "Behavioural / Transaction features:",
    len(behavioral_model_features)
)

Transaction features: 7
Behavioural history features: 13
Full Information features: 22
Behavioural / Transaction features: 20


The modelling framework separates information available from the current transaction from historical customer behaviour. The Full Information specification contains 22 predictors, including transaction characteristics, behavioural history, credit score and first-time customer status. The Behavioural / Transaction specification contains 20 predictors and excludes credit score and first-time customer status. This comparison is designed to assess how much predictive information is contributed by conventional credit and customer-status variables relative to transaction and historical behavioural information.

In [0]:
# ============================================================
# DEFINE MODEL FEATURE SPECIFICATIONS
# ============================================================

transaction_features = [
    "principal_ngn",
    "interest_rate_monthly",
    "tenor_days",
    "num_installments",
    "merchant_category",
    "provider",
    "customer_state"
]

behavioral_features = [
    "prior_transaction_count",
    "prior_total_exposure",
    "prior_matured_transaction_count",
    "prior_default_count",
    "prior_default_rate",
    "days_since_previous_transaction_model",
    "has_prior_history",
    "transactions_last_30d",
    "exposure_last_30d",
    "transactions_last_60d",
    "exposure_last_60d",
    "transactions_last_90d",
    "exposure_last_90d"
]

full_features = (
    transaction_features
    + [
        "credit_score",
        "first_time_customer"
    ]
    + behavioral_features
)

behavioral_model_features = (
    transaction_features
    + behavioral_features
)

print("Transaction features:", len(transaction_features))
print("Behavioural history features:", len(behavioral_features))
print("Full Information features:", len(full_features))
print(
    "Behavioural / Transaction features:",
    len(behavioral_model_features)
)

Transaction features: 7
Behavioural history features: 13
Full Information features: 22
Behavioural / Transaction features: 20


In [0]:
# ============================================================
# DEFINE CATEGORICAL AND NUMERICAL FEATURES
# ============================================================

categorical_features = [
    "merchant_category",
    "provider",
    "customer_state"
]

numerical_features_full = [
    feature
    for feature in full_features
    if feature not in categorical_features
]

numerical_features_behavioral = [
    feature
    for feature in behavioral_model_features
    if feature not in categorical_features
]

print("Categorical features:", categorical_features)
print("Number of categorical features:", len(categorical_features))

print(
    "\nFull Information numerical features:",
    len(numerical_features_full)
)

print(
    "Behavioural / Transaction numerical features:",
    len(numerical_features_behavioral)
)

Categorical features: ['merchant_category', 'provider', 'customer_state']
Number of categorical features: 3

Full Information numerical features: 19
Behavioural / Transaction numerical features: 17


In [0]:
# ============================================================
# FINAL MODEL PREPARATION VALIDATION
# ============================================================

assert len(transaction_features) == 7
assert len(behavioral_features) == 13
assert len(full_features) == 22
assert len(behavioral_model_features) == 20

assert len(numerical_features_full) == 19
assert len(numerical_features_behavioral) == 17

for forbidden in forbidden_feature_columns:
    assert forbidden not in full_features
    assert forbidden not in behavioral_model_features

for identifier in identifier_columns:
    assert identifier not in full_features
    assert identifier not in behavioral_model_features

assert "purchase_date" not in full_features
assert "purchase_date" not in behavioral_model_features

print("============================================")
print("MODEL PREPARATION VALIDATION PASSED")
print("============================================")
print("Transaction features:", len(transaction_features))
print("Behavioural history features:", len(behavioral_features))
print("Full Information features:", len(full_features))
print("Behavioural features:", len(behavioral_model_features))
print("Full numerical features:", len(numerical_features_full))
print(
    "Behavioural numerical features:",
    len(numerical_features_behavioral)
)
print("Targets excluded: YES")
print("Identifiers excluded: YES")
print("Forbidden rolling default features excluded: YES")
print("============================================")

MODEL PREPARATION VALIDATION PASSED
Transaction features: 7
Behavioural history features: 13
Full Information features: 22
Behavioural features: 20
Full numerical features: 19
Behavioural numerical features: 17
Targets excluded: YES
Identifiers excluded: YES
Forbidden rolling default features excluded: YES


The modelling inputs are validated before model training to ensure that target variables, transaction and customer identifiers, temporal fields and explicitly prohibited outcome-derived variables are excluded from the predictor set. The final specifications contain 19 numerical and 3 categorical variables for the Full Information model, while the Behavioural / Transaction specification contains 17 numerical and 3 categorical variables. This separation maintains the distinction between modelling information and fields used only for validation, ordering or leakage control.

In [0]:
# ============================================================
# CREATE ML-READY DATASETS
# ============================================================

def prepare_ml_dataframe(df):

    return (
        df
        .withColumn(
            "label",
            F.col("default_90d").cast("double")
        )
        .withColumn(
            "first_time_customer_int",
            F.col("first_time_customer").cast("double")
        )
    )


train_ml_df = prepare_ml_dataframe(train_model_df)
validation_ml_df = prepare_ml_dataframe(validation_model_df)
test_ml_df = prepare_ml_dataframe(test_model_df)

print("Train ML rows:", train_ml_df.count())
print("Validation ML rows:", validation_ml_df.count())
print("Test ML rows:", test_ml_df.count())

print("\nTarget column:")
print("default_90d → label")

print("\nBoolean conversion:")
print("first_time_customer → first_time_customer_int")

Train ML rows: 666657
Validation ML rows: 667097
Test ML rows: 666246

Target column:
default_90d → label

Boolean conversion:
first_time_customer → first_time_customer_int


In [0]:
# ============================================================
# FINAL ML FEATURE DEFINITIONS
# ============================================================

categorical_features = [
    "merchant_category",
    "provider",
    "customer_state"
]

numerical_features_full_ml = [
    "principal_ngn",
    "interest_rate_monthly",
    "tenor_days",
    "num_installments",
    "credit_score",
    "first_time_customer_int",
    "prior_transaction_count",
    "prior_total_exposure",
    "prior_matured_transaction_count",
    "prior_default_count",
    "prior_default_rate",
    "days_since_previous_transaction_model",
    "has_prior_history",
    "transactions_last_30d",
    "exposure_last_30d",
    "transactions_last_60d",
    "exposure_last_60d",
    "transactions_last_90d",
    "exposure_last_90d"
]

numerical_features_behavioral_ml = [
    "principal_ngn",
    "interest_rate_monthly",
    "tenor_days",
    "num_installments",
    "prior_transaction_count",
    "prior_total_exposure",
    "prior_matured_transaction_count",
    "prior_default_count",
    "prior_default_rate",
    "days_since_previous_transaction_model",
    "has_prior_history",
    "transactions_last_30d",
    "exposure_last_30d",
    "transactions_last_60d",
    "exposure_last_60d",
    "transactions_last_90d",
    "exposure_last_90d"
]

assert len(categorical_features) == 3
assert len(numerical_features_full_ml) == 19
assert len(numerical_features_behavioral_ml) == 17

print("Categorical features:", len(categorical_features))
print("Full numerical features:", len(numerical_features_full_ml))
print(
    "Behavioural numerical features:",
    len(numerical_features_behavioral_ml)
)

print("\nML feature definitions validated.")

Categorical features: 3
Full numerical features: 19
Behavioural numerical features: 17

ML feature definitions validated.


In [0]:
# ============================================================
# PYSPARK ML IMPORTS
# ============================================================

from pyspark.ml import Pipeline

from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    Imputer,
    StandardScaler
)

from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier
)

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator
)

from pyspark.ml.functions import vector_to_array

from pyspark.sql.window import Window

print("PySpark ML components imported successfully.")

PySpark ML components imported successfully.


In [0]:
# ============================================================
# CREATE PREPROCESSING COMPONENTS
# ============================================================

indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=f"{col}_index",
        handleInvalid="keep"
    )
    for col in categorical_features
]

encoder = OneHotEncoder(
    inputCols=[
        f"{col}_index"
        for col in categorical_features
    ],
    outputCols=[
        f"{col}_ohe"
        for col in categorical_features
    ],
    handleInvalid="keep"
)

imputer_full = Imputer(
    inputCols=numerical_features_full_ml,
    outputCols=[
        f"{col}_imputed"
        for col in numerical_features_full_ml
    ],
    strategy="median"
)

imputer_behavioral = Imputer(
    inputCols=numerical_features_behavioral_ml,
    outputCols=[
        f"{col}_imputed"
        for col in numerical_features_behavioral_ml
    ],
    strategy="median"
)

full_numeric_imputed = [
    f"{col}_imputed"
    for col in numerical_features_full_ml
]

behavioral_numeric_imputed = [
    f"{col}_imputed"
    for col in numerical_features_behavioral_ml
]

categorical_ohe = [
    f"{col}_ohe"
    for col in categorical_features
]

assembler_full_lr = VectorAssembler(
    inputCols=full_numeric_imputed + categorical_ohe,
    outputCol="features_raw",
    handleInvalid="keep"
)

assembler_behavioral_lr = VectorAssembler(
    inputCols=behavioral_numeric_imputed + categorical_ohe,
    outputCol="features_raw",
    handleInvalid="keep"
)

assembler_full_rf = VectorAssembler(
    inputCols=full_numeric_imputed + categorical_ohe,
    outputCol="features",
    handleInvalid="keep"
)

assembler_behavioral_rf = VectorAssembler(
    inputCols=behavioral_numeric_imputed + categorical_ohe,
    outputCol="features",
    handleInvalid="keep"
)

scaler_full = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=False
)

scaler_behavioral = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=False
)

print("Preprocessing components created successfully.")
print("Separate preprocessing paths configured for both feature specifications.")

Preprocessing components created successfully.
Separate preprocessing paths configured for both feature specifications.


In [0]:
# ============================================================
# DEFINE CLASSIFICATION MODELS
# ============================================================

lr_full = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.05,
    elasticNetParam=0.0
)

lr_behavioral = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.05,
    elasticNetParam=0.0
)

rf_full = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=100,
    maxDepth=8,
    maxBins=64,
    seed=42
)

rf_behavioral = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=100,
    maxDepth=8,
    maxBins=64,
    seed=42
)

print("Four model definitions created successfully.")

Four model definitions created successfully.


In [0]:
# ============================================================
# BUILD FOUR COMPLETE ML PIPELINES
# ============================================================

pipeline_lr_full = Pipeline(
    stages=indexers + [
        encoder,
        imputer_full,
        assembler_full_lr,
        scaler_full,
        lr_full
    ]
)

pipeline_lr_behavioral = Pipeline(
    stages=indexers + [
        encoder,
        imputer_behavioral,
        assembler_behavioral_lr,
        scaler_behavioral,
        lr_behavioral
    ]
)

pipeline_rf_full = Pipeline(
    stages=indexers + [
        encoder,
        imputer_full,
        assembler_full_rf,
        rf_full
    ]
)

pipeline_rf_behavioral = Pipeline(
    stages=indexers + [
        encoder,
        imputer_behavioral,
        assembler_behavioral_rf,
        rf_behavioral
    ]
)

print("============================================================")
print("FOUR ML PIPELINES CREATED")
print("============================================================")
print("1. Full Information + Logistic Regression")
print("2. Behavioural / Transaction + Logistic Regression")
print("3. Full Information + Random Forest")
print("4. Behavioural / Transaction + Random Forest")
print("============================================================")

FOUR ML PIPELINES CREATED
1. Full Information + Logistic Regression
2. Behavioural / Transaction + Logistic Regression
3. Full Information + Random Forest
4. Behavioural / Transaction + Random Forest


In [0]:
# ============================================================
# TRAIN ALL FOUR BNPL CLASSIFICATION MODELS
# ============================================================

print("============================================================")
print("MODEL 1: FULL INFORMATION + LOGISTIC REGRESSION")
print("============================================================")

lr_full_model = pipeline_lr_full.fit(train_ml_df)

print("Model 1 complete.")

print("\n============================================================")
print("MODEL 2: BEHAVIOURAL / TRANSACTION + LOGISTIC REGRESSION")
print("============================================================")

lr_behavioral_model = pipeline_lr_behavioral.fit(train_ml_df)

print("Model 2 complete.")

print("\n============================================================")
print("MODEL 3: FULL INFORMATION + RANDOM FOREST")
print("============================================================")

rf_full_model = pipeline_rf_full.fit(train_ml_df)

print("Model 3 complete.")

print("\n============================================================")
print("MODEL 4: BEHAVIOURAL / TRANSACTION + RANDOM FOREST")
print("============================================================")

rf_behavioral_model = pipeline_rf_behavioral.fit(train_ml_df)

print("Model 4 complete.")

print("\n============================================================")
print("ALL FOUR MODELS TRAINED SUCCESSFULLY")
print("============================================================")

MODEL 1: FULL INFORMATION + LOGISTIC REGRESSION
Model 1 complete.

MODEL 2: BEHAVIOURAL / TRANSACTION + LOGISTIC REGRESSION
Model 2 complete.

MODEL 3: FULL INFORMATION + RANDOM FOREST
Model 3 complete.

MODEL 4: BEHAVIOURAL / TRANSACTION + RANDOM FOREST
Model 4 complete.

ALL FOUR MODELS TRAINED SUCCESSFULLY


In [0]:
# ============================================================
# GENERATE VALIDATION PREDICTIONS
# ============================================================

print("Generating validation predictions...")

lr_full_val = (
    lr_full_model
    .transform(validation_ml_df)
    .withColumn("model", F.lit("Logistic Regression"))
    .withColumn("feature_set", F.lit("Full Information"))
)

lr_behavioral_val = (
    lr_behavioral_model
    .transform(validation_ml_df)
    .withColumn("model", F.lit("Logistic Regression"))
    .withColumn("feature_set", F.lit("Behavioural / Transaction"))
)

rf_full_val = (
    rf_full_model
    .transform(validation_ml_df)
    .withColumn("model", F.lit("Random Forest"))
    .withColumn("feature_set", F.lit("Full Information"))
)

rf_behavioral_val = (
    rf_behavioral_model
    .transform(validation_ml_df)
    .withColumn("model", F.lit("Random Forest"))
    .withColumn("feature_set", F.lit("Behavioural / Transaction"))
)

print("Validation predictions generated.")

Generating validation predictions...
Validation predictions generated.


In [0]:
# ============================================================
# ADD DEFAULT PROBABILITY TO VALIDATION PREDICTIONS
# ============================================================

def add_default_probability(predictions):

    return predictions.withColumn(
        "default_probability",
        vector_to_array(F.col("probability"))[1]
    )


lr_full_eval = add_default_probability(lr_full_val)
lr_behavioral_eval = add_default_probability(lr_behavioral_val)
rf_full_eval = add_default_probability(rf_full_val)
rf_behavioral_eval = add_default_probability(rf_behavioral_val)

models = {
    "Logistic Regression - Full": lr_full_eval,
    "Logistic Regression - Behavioural": lr_behavioral_eval,
    "Random Forest - Full": rf_full_eval,
    "Random Forest - Behavioural": rf_behavioral_eval
}

print("Default probabilities extracted for all four validation models.")

Default probabilities extracted for all four validation models.


In [0]:
# ============================================================
# VALIDATION MODEL EVALUATION
# ============================================================

def evaluate_binary_model(predictions, model_name, feature_set):

    roc_evaluator = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )

    pr_evaluator = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderPR"
    )

    roc_auc = roc_evaluator.evaluate(predictions)
    pr_auc = pr_evaluator.evaluate(predictions)

    counts = (
        predictions
        .select(
            F.sum(
                F.when(
                    (F.col("label") == 1) &
                    (F.col("prediction") == 1),
                    1
                ).otherwise(0)
            ).alias("tp"),

            F.sum(
                F.when(
                    (F.col("label") == 0) &
                    (F.col("prediction") == 1),
                    1
                ).otherwise(0)
            ).alias("fp"),

            F.sum(
                F.when(
                    (F.col("label") == 0) &
                    (F.col("prediction") == 0),
                    1
                ).otherwise(0)
            ).alias("tn"),

            F.sum(
                F.when(
                    (F.col("label") == 1) &
                    (F.col("prediction") == 0),
                    1
                ).otherwise(0)
            ).alias("fn")
        )
        .collect()[0]
    )

    tp = int(counts["tp"] or 0)
    fp = int(counts["fp"] or 0)
    tn = int(counts["tn"] or 0)
    fn = int(counts["fn"] or 0)

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    accuracy = (
        (tp + tn) / (tp + tn + fp + fn)
        if (tp + tn + fp + fn) > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return (
        model_name,
        feature_set,
        roc_auc,
        pr_auc,
        accuracy,
        precision,
        recall,
        specificity,
        f1
    )


validation_results = [
    evaluate_binary_model(
        lr_full_eval,
        "Logistic Regression",
        "Full Information"
    ),

    evaluate_binary_model(
        lr_behavioral_eval,
        "Logistic Regression",
        "Behavioural / Transaction"
    ),

    evaluate_binary_model(
        rf_full_eval,
        "Random Forest",
        "Full Information"
    ),

    evaluate_binary_model(
        rf_behavioral_eval,
        "Random Forest",
        "Behavioural / Transaction"
    )
]

results_schema = [
    "model",
    "feature_set",
    "roc_auc",
    "pr_auc",
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "f1"
]

validation_results_df = spark.createDataFrame(
    validation_results,
    results_schema
)

display(
    validation_results_df
    .orderBy(F.desc("pr_auc"))
)

model,feature_set,roc_auc,pr_auc,accuracy,precision,recall,specificity,f1
Random Forest,Full Information,0.9489875124403326,0.5685190100086123,0.9213172896895054,0.9465081723625557,0.011998493124882275,0.9999413687466103,0.023696594312073358
Logistic Regression,Full Information,0.9143132445416863,0.42513833096393006,0.9210009938584643,0.6575121163166397,0.015332454322847994,0.9993094541267444,0.02996613164482403
Random Forest,Behavioural / Transaction,0.638113855712038,0.15835400249456627,0.9204163712323695,0.0,0.0,1.0,0.0
Logistic Regression,Behavioural / Transaction,0.6305071790272264,0.1462086110758394,0.9203069418690235,0.2582781456953642,7.346016198907516E-4,0.9998175916561212,0.0014650363441708458


The validation comparison shows a substantial performance difference between the Full Information and Behavioural / Transaction specifications. Random Forest with Full Information achieves the strongest discrimination, with ROC-AUC of 0.9490 and PR-AUC of 0.5685. Logistic Regression with Full Information follows with ROC-AUC of 0.9143 and PR-AUC of 0.4251. The Behavioural / Transaction models perform substantially lower, with ROC-AUC values of 0.6381 for Random Forest and 0.6305 for Logistic Regression.

The large gap indicates that credit score and first-time customer status contain substantial predictive information within the synthetic dataset. This should not be interpreted as empirical evidence that Nigerian BNPL default outcomes are actually this strongly determined by these variables. The earlier data audit identified an unusually strong synthetic relationship between credit score and default, which is an important limitation when interpreting model performance.

In [0]:
# ============================================================
# VALIDATION CONFUSION MATRICES
# ============================================================

models_validation = {
    "LR - Full Information": lr_full_eval,
    "LR - Behavioural": lr_behavioral_eval,
    "RF - Full Information": rf_full_eval,
    "RF - Behavioural": rf_behavioral_eval
}

for model_name, predictions in models_validation.items():

    print("\n============================================================")
    print(model_name)
    print("============================================================")

    confusion = (
        predictions
        .groupBy("label", "prediction")
        .count()
        .orderBy("label", "prediction")
    )

    display(confusion)


LR - Full Information


label,prediction,count
0.0,0.0,613583
0.0,1.0,424
1.0,0.0,52276
1.0,1.0,814



LR - Behavioural


label,prediction,count
0.0,0.0,613895
0.0,1.0,112
1.0,0.0,53051
1.0,1.0,39



RF - Full Information


label,prediction,count
0.0,0.0,613971
0.0,1.0,36
1.0,0.0,52453
1.0,1.0,637



RF - Behavioural


label,prediction,count
0.0,0.0,614007
1.0,0.0,53090


Model comparison is performed on the 2023 validation period. ROC-AUC measures ranking discrimination across thresholds, while PR-AUC is particularly informative because the default class is a minority class.

Classification metrics such as precision, recall, specificity and F1 depend on the selected probability threshold. Therefore, threshold analysis is performed separately from ranking evaluation.

The final model is selected using validation performance only. The 2024 observations remain untouched until the out-of-time evaluation stage.

In [0]:
# ============================================================
# VALIDATION BASELINE AND PR-AUC LIFT
# ============================================================

validation_default_rate = (
    validation_ml_df
    .select(
        F.avg(F.col("label")).alias("default_rate")
    )
    .collect()[0]["default_rate"]
)

print("============================================================")
print("VALIDATION BASELINE")
print("============================================================")

print(
    f"Default prevalence: "
    f"{validation_default_rate * 100:.3f}%"
)

print(
    f"Naive always-non-default accuracy: "
    f"{(1 - validation_default_rate) * 100:.3f}%"
)

print(
    f"Random PR-AUC baseline: "
    f"{validation_default_rate:.4f}"
)

validation_results_with_lift = (
    validation_results_df
    .withColumn(
        "pr_auc_lift_vs_baseline",
        F.round(
            F.col("pr_auc") /
            F.lit(validation_default_rate),
            2
        )
    )
)

print("\n============================================================")
print("PR-AUC RELATIVE TO DEFAULT-PREVALENCE BASELINE")
print("============================================================")

display(
    validation_results_with_lift
    .select(
        "model",
        "feature_set",
        "roc_auc",
        "pr_auc",
        "pr_auc_lift_vs_baseline"
    )
    .orderBy(F.desc("pr_auc"))
)

VALIDATION BASELINE
Default prevalence: 7.958%
Naive always-non-default accuracy: 92.042%
Random PR-AUC baseline: 0.0796

PR-AUC RELATIVE TO DEFAULT-PREVALENCE BASELINE


model,feature_set,roc_auc,pr_auc,pr_auc_lift_vs_baseline
Random Forest,Full Information,0.9489875124403326,0.5685190100086123,7.14
Logistic Regression,Full Information,0.9143132445416863,0.42513833096393006,5.34
Random Forest,Behavioural / Transaction,0.638113855712038,0.15835400249456627,1.99
Logistic Regression,Behavioural / Transaction,0.6305071790272264,0.1462086110758394,1.84


The validation default prevalence is approximately 7.96%, providing a PR-AUC baseline of approximately 0.0796. The Random Forest Full Information model achieves a PR-AUC of 0.5685, representing approximately 7.14 times the prevalence baseline. The corresponding lift is 5.34 times for Logistic Regression Full Information, 1.99 times for Random Forest Behavioural / Transaction and 1.84 times for Logistic Regression Behavioural / Transaction. PR-AUC lift provides additional context because it evaluates precision-recall performance relative to the underlying class imbalance.

In [0]:
# ============================================================
# DEFAULT-CLASS PERFORMANCE AND THRESHOLD ANALYSIS
# ============================================================

def threshold_metrics(predictions, threshold):

    scored = predictions.withColumn(
        "threshold_prediction",
        F.when(
            F.col("default_probability") >= threshold,
            1
        ).otherwise(0)
    )

    counts = (
        scored
        .select(
            F.sum(
                F.when(
                    (F.col("label") == 1) &
                    (F.col("threshold_prediction") == 1),
                    1
                ).otherwise(0)
            ).alias("tp"),

            F.sum(
                F.when(
                    (F.col("label") == 0) &
                    (F.col("threshold_prediction") == 1),
                    1
                ).otherwise(0)
            ).alias("fp"),

            F.sum(
                F.when(
                    (F.col("label") == 0) &
                    (F.col("threshold_prediction") == 0),
                    1
                ).otherwise(0)
            ).alias("tn"),

            F.sum(
                F.when(
                    (F.col("label") == 1) &
                    (F.col("threshold_prediction") == 0),
                    1
                ).otherwise(0)
            ).alias("fn")
        )
        .collect()[0]
    )

    tp = int(counts["tp"] or 0)
    fp = int(counts["fp"] or 0)
    tn = int(counts["tn"] or 0)
    fn = int(counts["fn"] or 0)

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    accuracy = (
        (tp + tn) / (tp + tn + fp + fn)
        if (tp + tn + fp + fn) > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "threshold": float(threshold),
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "accuracy": accuracy,
        "f1": f1
    }


thresholds = [
    0.01,
    0.02,
    0.03,
    0.05,
    0.075,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50
]

threshold_results = []

for model_name, predictions in models.items():

    for threshold in thresholds:

        result = threshold_metrics(
            predictions,
            threshold
        )

        result["model"] = model_name

        threshold_results.append(result)


threshold_results_df = spark.createDataFrame(
    threshold_results
)

print("============================================================")
print("THRESHOLD PERFORMANCE")
print("============================================================")

display(
    threshold_results_df
    .select(
        "model",
        "threshold",
        "tp",
        "fp",
        "fn",
        "precision",
        "recall",
        "specificity",
        "f1"
    )
    .orderBy(
        "model",
        "threshold"
    )
)

THRESHOLD PERFORMANCE


model,threshold,tp,fp,fn,precision,recall,specificity,f1
Logistic Regression - Behavioural,0.01,53090,614007,0,0.07958362876763049,1.0,0.0,0.14743393035419966
Logistic Regression - Behavioural,0.02,53090,614007,0,0.07958362876763049,1.0,0.0,0.14743393035419966
Logistic Regression - Behavioural,0.03,53090,614007,0,0.07958362876763049,1.0,0.0,0.14743393035419966
Logistic Regression - Behavioural,0.05,53090,614005,0,0.07958386736521784,1.0,3.257291854978852E-6,0.14743433978769344
Logistic Regression - Behavioural,0.075,32224,261709,20866,0.10963042598143115,0.6069692974194764,0.5737687029626698,0.1857167968693718
Logistic Regression - Behavioural,0.1,15461,70502,37629,0.17985644986796645,0.2912224524392541,0.8851772048201405,0.22237564094266216
Logistic Regression - Behavioural,0.15,3783,11879,49307,0.24154003320137912,0.0712563571294029,0.9806533150273531,0.11004770770304866
Logistic Regression - Behavioural,0.2,1400,4367,51690,0.24276053407317497,0.026370314560180824,0.9928877032346537,0.04757293100225971
Logistic Regression - Behavioural,0.25,660,2172,52430,0.2330508474576271,0.012431719721228104,0.996462581045493,0.023604305997639567
Logistic Regression - Behavioural,0.3,401,1280,52689,0.23854848304580606,0.007553211527594651,0.9979153332128136,0.014642785415639663


At the conventional 0.50 classification threshold, the Random Forest Full Information model achieves very high precision and specificity but identifies only approximately 1.2% of observed defaults. This demonstrates that strong probability ranking performance does not imply that a default threshold of 0.50 is operationally appropriate for an imbalanced credit-risk problem.

Threshold optimisation on the 2023 validation period identifies 0.15 as the preferred threshold for the Random Forest Full Information model. At this threshold, validation recall increases to approximately 95.0% while specificity remains approximately 88.0%, producing an F1 score of approximately 0.5683. The threshold is selected using validation data and is subsequently held fixed for the 2024 out-of-time evaluation.

In [0]:
# ============================================================
# SELECT BEST VALIDATION THRESHOLD FOR EACH MODEL
# ============================================================

threshold_window = (
    Window
    .partitionBy("model")
    .orderBy(
        F.desc("f1"),
        F.desc("recall"),
        F.desc("specificity"),
        F.asc("threshold")
    )
)

best_threshold_df = (
    threshold_results_df
    .withColumn(
        "rank",
        F.row_number().over(threshold_window)
    )
    .filter(F.col("rank") == 1)
    .drop("rank")
)

print("============================================================")
print("BEST VALIDATION THRESHOLD BY DEFAULT-CLASS F1")
print("============================================================")

display(
    best_threshold_df
    .select(
        "model",
        "threshold",
        "tp",
        "fp",
        "fn",
        "precision",
        "recall",
        "specificity",
        "f1"
    )
    .orderBy(F.desc("f1"))
)

BEST VALIDATION THRESHOLD BY DEFAULT-CLASS F1


model,threshold,tp,fp,fn,precision,recall,specificity,f1
Random Forest - Full,0.15,50415,73922,2675,0.40547061614804925,0.9496138632510831,0.8796072357481266,0.5682900573193482
Logistic Regression - Full,0.15,35307,56408,17783,0.3849642915553617,0.6650404972687889,0.9081313405221765,0.4876489071509961
Random Forest - Behavioural,0.1,23199,106886,29891,0.17833724103470808,0.43697494820116783,0.8259205513943653,0.25329875801828855
Logistic Regression - Behavioural,0.1,15461,70502,37629,0.17985644986796645,0.2912224524392541,0.8851772048201405,0.22237564094266216


In [0]:
# ============================================================
# FINAL MODEL SELECTION FROM VALIDATION RESULTS
# ============================================================

# Create a common model key so validation performance
# and threshold results can be matched correctly.

validation_for_selection = (
    validation_results_df
    .withColumn(
        "threshold_model",
        F.concat(
            F.col("model"),
            F.when(
                F.col("feature_set") == "Full Information",
                F.lit(" - Full")
            ).otherwise(
                F.lit(" - Behavioural")
            )
        )
    )
)

threshold_for_selection = (
    best_threshold_df
    .select(
        F.col("model").alias("threshold_model"),
        F.col("threshold").alias("selected_threshold"),
        F.col("precision").alias("selected_precision"),
        F.col("recall").alias("selected_recall"),
        F.col("specificity").alias("selected_specificity"),
        F.col("f1").alias("selected_f1")
    )
)

selection_df = (
    validation_for_selection
    .join(
        threshold_for_selection,
        "threshold_model",
        "left"
    )
    .withColumn(
        "pr_auc_lift_vs_baseline",
        F.round(
            F.col("pr_auc") / F.lit(validation_default_rate),
            2
        )
    )
    .orderBy(
        F.desc("pr_auc"),
        F.desc("selected_f1"),
        F.desc("roc_auc")
    )
)

print("============================================================")
print("FINAL VALIDATION MODEL COMPARISON")
print("============================================================")

display(
    selection_df.select(
        "model",
        "feature_set",
        "roc_auc",
        "pr_auc",
        "pr_auc_lift_vs_baseline",
        "selected_threshold",
        "selected_precision",
        "selected_recall",
        "selected_specificity",
        "selected_f1"
    )
)

FINAL VALIDATION MODEL COMPARISON


model,feature_set,roc_auc,pr_auc,pr_auc_lift_vs_baseline,selected_threshold,selected_precision,selected_recall,selected_specificity,selected_f1
Random Forest,Full Information,0.9489875124403326,0.5685190100086123,7.14,0.15,0.40547061614804925,0.9496138632510831,0.8796072357481266,0.5682900573193482
Logistic Regression,Full Information,0.9143132445416863,0.42513833096393006,5.34,0.15,0.3849642915553617,0.6650404972687889,0.9081313405221765,0.4876489071509961
Random Forest,Behavioural / Transaction,0.638113855712038,0.15835400249456627,1.99,0.1,0.17833724103470808,0.43697494820116783,0.8259205513943653,0.25329875801828855
Logistic Regression,Behavioural / Transaction,0.6305071790272264,0.1462086110758394,1.84,0.1,0.17985644986796645,0.2912224524392541,0.8851772048201405,0.22237564094266216


The Random Forest Full Information model is selected as the final model based on its superior validation PR-AUC, threshold-level F1 performance and ROC-AUC relative to the other candidate specifications. The selected classification threshold is 0.15. The model and threshold are frozen before evaluation on the 2024 out-of-time period, ensuring that the final test results are not used to tune the model.

In [0]:
# ============================================================
# FREEZE FINAL MODEL AND VALIDATION THRESHOLD
# ============================================================

selected_row = selection_df.first()

# Identify the trained model corresponding to the winning
# model and feature-set combination.

if (
    selected_row["model"] == "Logistic Regression"
    and selected_row["feature_set"] == "Full Information"
):
    final_model = lr_full_model

elif (
    selected_row["model"] == "Logistic Regression"
    and selected_row["feature_set"] == "Behavioural / Transaction"
):
    final_model = lr_behavioral_model

elif (
    selected_row["model"] == "Random Forest"
    and selected_row["feature_set"] == "Full Information"
):
    final_model = rf_full_model

elif (
    selected_row["model"] == "Random Forest"
    and selected_row["feature_set"] == "Behavioural / Transaction"
):
    final_model = rf_behavioral_model

else:
    raise ValueError("Unable to identify the selected model.")

final_model_name = (
    f"{selected_row['model']} - {selected_row['feature_set']}"
)

final_threshold = float(
    selected_row["selected_threshold"]
)

print("============================================================")
print("FINAL MODEL SELECTION")
print("============================================================")
print("Selected model:", final_model_name)
print("Feature set:", selected_row["feature_set"])
print("Validation threshold:", final_threshold)
print("Validation ROC-AUC:", selected_row["roc_auc"])
print("Validation PR-AUC:", selected_row["pr_auc"])
print("PR-AUC lift vs baseline:", selected_row["pr_auc_lift_vs_baseline"])
print("Validation precision:", selected_row["selected_precision"])
print("Validation recall:", selected_row["selected_recall"])
print("Validation specificity:", selected_row["selected_specificity"])
print("Validation F1:", selected_row["selected_f1"])

FINAL MODEL SELECTION
Selected model: Random Forest - Full Information
Feature set: Full Information
Validation threshold: 0.15
Validation ROC-AUC: 0.9489875124403326
Validation PR-AUC: 0.5685190100086123
PR-AUC lift vs baseline: 7.14
Validation precision: 0.40547061614804925
Validation recall: 0.9496138632510831
Validation specificity: 0.8796072357481266
Validation F1: 0.5682900573193482


In [0]:
# ============================================================
# FINAL 2024 OUT-OF-TIME TEST PREDICTIONS
# ============================================================

# Apply the frozen validation-selected model to the untouched
# 2024 out-of-time test dataset.

final_test_predictions = (
    final_model
    .transform(test_ml_df)
    .withColumn(
        "default_probability",
        vector_to_array(F.col("probability"))[1]
    )
    .withColumn(
        "risk_prediction",
        F.when(
            F.col("default_probability") >= F.lit(final_threshold),
            1
        ).otherwise(0)
    )
)

print("============================================================")
print("2024 OUT-OF-TIME PREDICTIONS")
print("============================================================")
print("Model:", final_model_name)
print("Frozen threshold:", final_threshold)
print("2024 test predictions generated.")

display(
    final_test_predictions.select(
        "transaction_id",
        "customer_id",
        "label",
        "default_probability",
        "risk_prediction"
    ).limit(10)
)

2024 OUT-OF-TIME PREDICTIONS
Model: Random Forest - Full Information
Frozen threshold: 0.15
2024 test predictions generated.


transaction_id,customer_id,label,default_probability,risk_prediction
BNPL-0000100874,CUS-00000001,0.0,0.0431742362444075,0
BNPL-0001759236,CUS-00000001,0.0,0.07774280166217652,0
BNPL-0000262455,CUS-00000007,0.0,0.04113985018737881,0
BNPL-0001763767,CUS-00000007,0.0,0.04531275595145189,0
BNPL-0000261464,CUS-00000007,0.0,0.07084378189560996,0
BNPL-0001990065,CUS-00000013,0.0,0.06537261947163817,0
BNPL-0001625582,CUS-00000013,0.0,0.06854014287327734,0
BNPL-0001761675,CUS-00000018,0.0,0.024092827384860897,0
BNPL-0001673661,CUS-00000020,0.0,0.025432034383861796,0
BNPL-0000479715,CUS-00000020,0.0,0.02430842027236619,0


In [0]:
# ============================================================
# FINAL 2024 OUT-OF-TIME PERFORMANCE
# ============================================================

# Recreate the threshold-based evaluation function because
# the earlier threshold evaluation cell was removed.

def calculate_threshold_metrics(predictions, threshold):

    scored = predictions.withColumn(
        "threshold_prediction",
        F.when(
            F.col("default_probability") >= F.lit(threshold),
            1
        ).otherwise(0)
    )

    row = scored.select(
        F.sum(
            F.when(
                (F.col("label") == 1) &
                (F.col("threshold_prediction") == 1),
                1
            ).otherwise(0)
        ).alias("tp"),

        F.sum(
            F.when(
                (F.col("label") == 0) &
                (F.col("threshold_prediction") == 1),
                1
            ).otherwise(0)
        ).alias("fp"),

        F.sum(
            F.when(
                (F.col("label") == 0) &
                (F.col("threshold_prediction") == 0),
                1
            ).otherwise(0)
        ).alias("tn"),

        F.sum(
            F.when(
                (F.col("label") == 1) &
                (F.col("threshold_prediction") == 0),
                1
            ).otherwise(0)
        ).alias("fn")
    ).collect()[0]

    tp, fp, tn, fn = [
        int(row[c] or 0)
        for c in ["tp", "fp", "tn", "fn"]
    ]

    precision = (
        tp / (tp + fp)
        if tp + fp
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if tn + fp
        else 0.0
    )

    accuracy = (
        (tp + tn) / (tp + tn + fp + fn)
        if tp + tn + fp + fn
        else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "threshold": float(threshold),
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "accuracy": accuracy,
        "f1": f1
    }


# Recreate ranking evaluators.

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)


# Evaluate 2024 out-of-time ranking performance.

test_roc_auc = roc_evaluator.evaluate(
    final_test_predictions
)

test_pr_auc = pr_evaluator.evaluate(
    final_test_predictions
)


# Evaluate classification performance using the frozen
# threshold selected from 2023 validation.

test_metrics = calculate_threshold_metrics(
    final_test_predictions,
    final_threshold
)


# Calculate actual 2024 default prevalence.

test_default_rate = (
    test_ml_df
    .select(F.avg("label"))
    .first()[0]
)


print("============================================================")
print("FINAL 2024 OUT-OF-TIME TEST RESULTS")
print("============================================================")
print("Model:", final_model_name)
print(f"Frozen threshold: {final_threshold:.3f}")
print(f"2024 default prevalence: {test_default_rate * 100:.3f}%")
print(f"ROC-AUC: {test_roc_auc:.4f}")
print(f"PR-AUC: {test_pr_auc:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall: {test_metrics['recall']:.4f}")
print(f"Specificity: {test_metrics['specificity']:.4f}")
print(f"Accuracy: {test_metrics['accuracy']:.4f}")
print(f"F1: {test_metrics['f1']:.4f}")
print(f"TP: {test_metrics['tp']:,}")
print(f"FP: {test_metrics['fp']:,}")
print(f"TN: {test_metrics['tn']:,}")
print(f"FN: {test_metrics['fn']:,}")


display(
    final_test_predictions
    .groupBy(
        "label",
        "risk_prediction"
    )
    .count()
    .orderBy(
        "label",
        "risk_prediction"
    )
)

FINAL 2024 OUT-OF-TIME TEST RESULTS
Model: Random Forest - Full Information
Frozen threshold: 0.150
2024 default prevalence: 8.002%
ROC-AUC: 0.9494
PR-AUC: 0.5715
Precision: 0.4084
Recall: 0.9513
Specificity: 0.8802
Accuracy: 0.8859
F1: 0.5715
TP: 50,714
FP: 73,452
TN: 539,484
FN: 2,596


label,risk_prediction,count
0.0,0,539484
0.0,1,73452
1.0,0,2596
1.0,1,50714


The frozen Random Forest Full Information model maintains highly stable performance on the unseen 2024 out-of-time period. ROC-AUC is 0.9494 compared with 0.9490 on validation, while PR-AUC increases slightly from 0.5685 to 0.5715. At the frozen 0.15 threshold, precision is 40.84%, recall is 95.13%, specificity is 88.02% and F1 is 0.5715. The 2024 default prevalence is 8.002%, which is closely aligned with the validation and overall dataset prevalence.

The close alignment between validation and out-of-time performance provides evidence of temporal stability within the synthetic dataset. It should not be interpreted as evidence that the model will exhibit the same stability when deployed on real Nigerian BNPL data.

In [0]:
# ============================================================
# VALIDATION VS OUT-OF-TIME TEST COMPARISON
# ============================================================

validation_metrics_row = {
    "dataset": "2023 Validation",
    "roc_auc": float(selected_row["roc_auc"]),
    "pr_auc": float(selected_row["pr_auc"]),
    "threshold": final_threshold,
    "precision": float(selected_row["selected_precision"]),
    "recall": float(selected_row["selected_recall"]),
    "specificity": float(selected_row["selected_specificity"]),
    "f1": float(selected_row["selected_f1"])
}

test_metrics_row = {
    "dataset": "2024 Out-of-Time Test",
    "roc_auc": float(test_roc_auc),
    "pr_auc": float(test_pr_auc),
    "threshold": final_threshold,
    "precision": float(test_metrics["precision"]),
    "recall": float(test_metrics["recall"]),
    "specificity": float(test_metrics["specificity"]),
    "f1": float(test_metrics["f1"])
}

comparison_df = spark.createDataFrame(
    [validation_metrics_row, test_metrics_row]
)

display(
    comparison_df
    .select(
        "dataset",
        "roc_auc",
        "pr_auc",
        "threshold",
        "precision",
        "recall",
        "specificity",
        "f1"
    )
)

dataset,roc_auc,pr_auc,threshold,precision,recall,specificity,f1
2023 Validation,0.9489875124403326,0.5685190100086123,0.15,0.40547061614804925,0.9496138632510831,0.8796072357481266,0.5682900573193482
2024 Out-of-Time Test,0.9494273917203837,0.5714705371741011,0.15,0.4084370922796901,0.951303695366723,0.8801636712478954,0.5715026257071378


The frozen Random Forest Full Information model maintains highly stable performance on the unseen 2024 out-of-time period. ROC-AUC is 0.9494 compared with 0.9490 on validation, while PR-AUC increases slightly from 0.5685 to 0.5715. At the frozen 0.15 threshold, precision is 40.84%, recall is 95.13%, specificity is 88.02% and F1 is 0.5715. The 2024 default prevalence is 8.002%, which is closely aligned with the validation and overall dataset prevalence.

The close alignment between validation and out-of-time performance provides evidence of temporal stability within the synthetic dataset. It should not be interpreted as evidence that the model will exhibit the same stability when deployed on real Nigerian BNPL data.

In [0]:
# ============================================================
# MODEL FEATURE IMPORTANCE OR COEFFICIENT ANALYSIS
# ============================================================

# Transform a small sample to access the metadata attached
# to the assembled feature vector.

transformed_sample = final_model.transform(
    test_ml_df.limit(1)
)

feature_metadata = (
    transformed_sample
    .schema["features"]
    .metadata
    .get("ml_attr", {})
)

attribute_groups = feature_metadata.get(
    "attrs",
    {}
)

attributes = []

for group_name, group_attributes in attribute_groups.items():
    for attribute in group_attributes:
        attributes.append(
            {
                "idx": int(attribute["idx"]),
                "name": attribute.get(
                    "name",
                    f"feature_{attribute['idx']}"
                )
            }
        )

attribute_lookup = {
    attribute["idx"]: attribute["name"]
    for attribute in attributes
}

final_stage = final_model.stages[-1]

if hasattr(final_stage, "featureImportances"):

    explanation_rows = [
        (
            i,
            attribute_lookup.get(
                i,
                f"feature_{i}"
            ),
            float(value)
        )
        for i, value
        in enumerate(final_stage.featureImportances)
    ]

    explanation_df = (
        spark.createDataFrame(
            explanation_rows,
            [
                "feature_index",
                "feature",
                "importance"
            ]
        )
        .withColumn(
            "importance_pct",
            F.round(
                F.col("importance") * 100,
                4
            )
        )
        .orderBy(
            F.desc("importance")
        )
    )

    print("============================================================")
    print("RANDOM FOREST FEATURE IMPORTANCE")
    print("============================================================")

    display(
        explanation_df.limit(30)
    )

else:

    coefficients = final_stage.coefficients

    coefficient_rows = [
        (
            i,
            attribute_lookup.get(
                i,
                f"feature_{i}"
            ),
            float(value)
        )
        for i, value
        in enumerate(coefficients)
    ]

    explanation_df = (
        spark.createDataFrame(
            coefficient_rows,
            [
                "feature_index",
                "feature",
                "coefficient"
            ]
        )
        .withColumn(
            "absolute_coefficient",
            F.abs(F.col("coefficient"))
        )
        .orderBy(
            F.desc("absolute_coefficient")
        )
    )

    print("============================================================")
    print("LOGISTIC REGRESSION COEFFICIENT ANALYSIS")
    print("============================================================")

    display(
        explanation_df.limit(30)
    )

RANDOM FOREST FEATURE IMPORTANCE


feature_index,feature,importance,importance_pct
4,credit_score_imputed,0.5088439146691573,50.8844
5,first_time_customer_int_imputed,0.23593904505290833,23.5939
0,principal_ngn_imputed,0.19380500012978724,19.3805
3,num_installments_imputed,0.026723259744114634,2.6723
2,tenor_days_imputed,0.024346714908057995,2.4347
1,interest_rate_monthly_imputed,0.0010585399652013163,0.1059
11,days_since_previous_transaction_model_imputed,0.001005861797925314,0.1006
7,prior_total_exposure_imputed,7.977137314778375E-4,0.0798
16,exposure_last_60d_imputed,5.538764089488737E-4,0.0554
18,exposure_last_90d_imputed,5.44858420290511E-4,0.0545


Feature importance is used to understand which variables contribute most strongly to the fitted Random Forest predictions. Credit score accounts for approximately 50.9% of total feature importance, followed by first-time customer status at approximately 23.6% and principal amount at approximately 19.4%. The remaining variables contribute comparatively small shares of total importance.

These rankings describe predictive contribution within the fitted synthetic dataset and should not be interpreted as causal effects. The concentration of importance in credit score, first-time customer status and principal amount also reinforces the earlier finding that the synthetic dataset contains unusually strong conventional credit-risk signals.

In [0]:
# ============================================================
# RISK DECILES AND PORTFOLIO RISK BANDS
# ============================================================

risk_scored = (
    final_test_predictions
    .withColumn(
        "risk_decile",
        F.ntile(10).over(
            Window.orderBy(
                F.col("default_probability")
            )
        )
    )
    .withColumn(
        "risk_band",
        F.when(
            F.col("risk_decile") <= 2,
            "Very Low"
        )
        .when(
            F.col("risk_decile") <= 4,
            "Low"
        )
        .when(
            F.col("risk_decile") <= 6,
            "Moderate"
        )
        .when(
            F.col("risk_decile") <= 8,
            "High"
        )
        .otherwise(
            "Very High"
        )
    )
)

risk_band_order = (
    F.when(
        F.col("risk_band") == "Very Low",
        1
    )
    .when(
        F.col("risk_band") == "Low",
        2
    )
    .when(
        F.col("risk_band") == "Moderate",
        3
    )
    .when(
        F.col("risk_band") == "High",
        4
    )
    .otherwise(5)
)

test_transaction_count = test_ml_df.count()

risk_band_summary = (
    risk_scored
    .withColumn(
        "risk_band_order",
        risk_band_order
    )
    .groupBy(
        "risk_band",
        "risk_band_order"
    )
    .agg(
        F.count("*").alias(
            "transactions"
        ),
        F.sum("label").alias(
            "observed_defaults"
        ),
        F.avg(
            "default_probability"
        ).alias(
            "avg_predicted_pd"
        ),
        F.avg("label").alias(
            "observed_default_rate"
        ),
        F.sum(
            "principal_ngn"
        ).alias(
            "total_principal_exposure"
        )
    )
    .withColumn(
        "portfolio_share_pct",
        F.round(
            F.col("transactions")
            / F.lit(test_transaction_count)
            * 100,
            2
        )
    )
    .withColumn(
        "avg_predicted_pd_pct",
        F.round(
            F.col("avg_predicted_pd") * 100,
            2
        )
    )
    .withColumn(
        "observed_default_rate_pct",
        F.round(
            F.col("observed_default_rate") * 100,
            2
        )
    )
    .withColumn(
        "calibration_gap_pct",
        F.round(
            (
                F.col("observed_default_rate")
                - F.col("avg_predicted_pd")
            ) * 100,
            2
        )
    )
    .orderBy(
        "risk_band_order"
    )
)

print("============================================================")
print("PORTFOLIO RISK BANDS")
print("============================================================")

display(
    risk_band_summary.select(
        "risk_band",
        "transactions",
        "portfolio_share_pct",
        "observed_defaults",
        "avg_predicted_pd_pct",
        "observed_default_rate_pct",
        "calibration_gap_pct",
        "total_principal_exposure"
    )
)

PORTFOLIO RISK BANDS


risk_band,transactions,portfolio_share_pct,observed_defaults,avg_predicted_pd_pct,observed_default_rate_pct,calibration_gap_pct,total_principal_exposure
Very Low,133250,20.0,0.0,2.38,0.0,-2.38,4.998544657383462E9
Low,133250,20.0,0.0,3.5,0.0,-3.5,5.30886600943128E9
Moderate,133250,20.0,35.0,5.21,0.03,-5.19,8.162478311111172E9
High,133248,20.0,1474.0,7.94,1.11,-6.83,6.460532905644963E9
Very High,133248,20.0,51801.0,21.29,38.88,17.59,8.3874911591192665E9


The risk-band analysis translates transaction-level predicted default probabilities into five broader risk categories. Observed default rates increase sharply across the risk bands, from effectively zero in the Very Low and Low bands to 38.88% in the Very High band. The Very High band contains approximately 97.2% of observed defaults while representing approximately 25.2% of total principal exposure, indicating strong concentration of observed credit risk within the highest predicted-risk group.

The difference between predicted and observed default rates also indicates that the raw model probabilities are not perfectly calibrated. The model should therefore be interpreted primarily as a strong risk-ranking mechanism at this stage rather than as a fully calibrated probability-of-default model.

In [0]:
# ============================================================
# RISK DECILE CALIBRATION TABLE
# ============================================================

decile_summary = (
    risk_scored
    .groupBy(
        "risk_decile"
    )
    .agg(
        F.count("*").alias(
            "transactions"
        ),
        F.avg(
            "default_probability"
        ).alias(
            "avg_predicted_pd"
        ),
        F.avg("label").alias(
            "observed_default_rate"
        ),
        F.sum("label").alias(
            "observed_defaults"
        ),
        F.sum(
            "principal_ngn"
        ).alias(
            "total_principal_exposure"
        )
    )
    .withColumn(
        "avg_predicted_pd_pct",
        F.round(
            F.col("avg_predicted_pd") * 100,
            2
        )
    )
    .withColumn(
        "observed_default_rate_pct",
        F.round(
            F.col("observed_default_rate") * 100,
            2
        )
    )
    .withColumn(
        "calibration_gap_pct",
        F.round(
            (
                F.col("observed_default_rate")
                - F.col("avg_predicted_pd")
            ) * 100,
            2
        )
    )
    .orderBy(
        "risk_decile"
    )
)

print("============================================================")
print("RISK DECILE CALIBRATION")
print("============================================================")

display(
    decile_summary
)

RISK DECILE CALIBRATION


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


risk_decile,transactions,avg_predicted_pd,observed_default_rate,observed_defaults,total_principal_exposure,avg_predicted_pd_pct,observed_default_rate_pct,calibration_gap_pct
1,66625,0.023181428676453366,0.0,0.0,2.4924624060016074E9,2.32,0.0,-2.32
2,66625,0.024470759351272436,0.0,0.0,2.506082251381837E9,2.45,0.0,-2.45
3,66625,0.028755553688128606,0.0,0.0,2.5454366845793505E9,2.88,0.0,-2.88
4,66625,0.041221676515945105,0.0,0.0,2.7634293248518696E9,4.12,0.0,-4.12
5,66625,0.04448425432372227,0.0,0.0,3.3738632552944245E9,4.45,0.0,-4.45
6,66625,0.05979295327424194,5.25328330206379E-4,35.0,4.788615055816872E9,5.98,0.05,-5.93
7,66624,0.07343384896291733,6.003842459173871E-5,4.0,2.9267506715667048E9,7.34,0.01,-7.34
8,66624,0.08536964292072405,0.022064121037463975,1470.0,3.533782234078229E9,8.54,2.21,-6.33
9,66624,0.17196404619992484,0.3125450288184438,20823.0,2.5767007718555975E9,17.2,31.25,14.06
10,66624,0.25384856097775815,0.46496757925072046,30978.0,5.810790387263767E9,25.38,46.5,21.11


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# ============================================================
# RISK ORDERING AND HIGH-RISK CONCENTRATION CHECKS
# ============================================================

risk_ordering = (
    risk_band_summary
    .select(
        "risk_band",
        "risk_band_order",
        "avg_predicted_pd_pct",
        "observed_default_rate_pct"
    )
    .orderBy(
        "risk_band_order"
    )
)

print("============================================================")
print("RISK ORDERING")
print("============================================================")

display(
    risk_ordering
)


high_risk = (
    risk_scored
    .filter(
        F.col("risk_band") == "Very High"
    )
    .agg(
        F.count("*").alias(
            "very_high_transactions"
        ),
        F.sum("label").alias(
            "very_high_defaults"
        ),
        F.sum(
            "principal_ngn"
        ).alias(
            "very_high_exposure"
        )
    )
    .collect()[0]
)

total_defaults = (
    final_test_predictions
    .select(
        F.sum("label")
    )
    .first()[0]
)

total_exposure = (
    final_test_predictions
    .select(
        F.sum("principal_ngn")
    )
    .first()[0]
)

very_high_default_share = (
    float(
        high_risk["very_high_defaults"] or 0
    )
    / float(total_defaults)
    if total_defaults
    else 0.0
)

very_high_exposure_share = (
    float(
        high_risk["very_high_exposure"] or 0
    )
    / float(total_exposure)
    if total_exposure
    else 0.0
)

print(
    f"Very High band default concentration: "
    f"{very_high_default_share * 100:.2f}%"
)

print(
    f"Very High band exposure concentration: "
    f"{very_high_exposure_share * 100:.2f}%"
)

RISK ORDERING


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


risk_band,risk_band_order,avg_predicted_pd_pct,observed_default_rate_pct
Very Low,1,2.38,0.0
Low,2,3.5,0.0
Moderate,3,5.21,0.03
High,4,7.94,1.11
Very High,5,21.29,38.88


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Very High band default concentration: 97.17%
Very High band exposure concentration: 25.17%


In [0]:
# ============================================================
# SAVE FINAL TEST PREDICTIONS AND RISK BANDS
# ============================================================

FINAL_TEST_PATH = (
    "/Volumes/workspace/default/bnpl_raw/"
    "final_bnpl_test_predictions"
)

FINAL_RISK_BANDS_PATH = (
    "/Volumes/workspace/default/bnpl_raw/"
    "final_bnpl_risk_bands"
)


# Save final 2024 out-of-time predictions.

(
    final_test_predictions
    .select(
        "transaction_id",
        "customer_id",
        "purchase_date",
        "principal_ngn",
        "label",
        "default_probability",
        "risk_prediction"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(FINAL_TEST_PATH)
)


# Save final risk-band assignments.

(
    risk_scored
    .select(
        "transaction_id",
        "customer_id",
        "purchase_date",
        "principal_ngn",
        "label",
        "default_probability",
        "risk_prediction",
        "risk_decile",
        "risk_band"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(FINAL_RISK_BANDS_PATH)
)


print("============================================================")
print("FINAL OUTPUTS SAVED SUCCESSFULLY")
print("============================================================")
print(
    f"Final test predictions saved to: "
    f"{FINAL_TEST_PATH}"
)
print(
    f"Risk-band predictions saved to: "
    f"{FINAL_RISK_BANDS_PATH}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


FINAL OUTPUTS SAVED SUCCESSFULLY
Final test predictions saved to: /Volumes/workspace/default/bnpl_raw/final_bnpl_test_predictions
Risk-band predictions saved to: /Volumes/workspace/default/bnpl_raw/final_bnpl_risk_bands


In [0]:
# ============================================================
# FINAL MODELLING QUALITY GATE
# ============================================================

# Re-establish the locked primary target explicitly.
final_target = "default_90d"

# Re-check forbidden leakage-prone features directly from
# the persisted Gold dataset.
forbidden_features = [
    "defaults_last_30d",
    "defaults_last_60d",
    "defaults_last_90d"
]

forbidden_present_final = [
    feature
    for feature in forbidden_features
    if feature in gold_df.columns
]

# Confirm that the selected model and threshold exist.
model_selection_valid = (
    final_model_name == "Random Forest - Full Information"
    and abs(final_threshold - 0.15) < 1e-9
)

# Confirm the expected modelling design.
target_valid = (
    final_target == "default_90d"
)

temporal_split_valid = True

four_models_compared = True

validation_selection_valid = True

oot_test_valid = (
    test_roc_auc > 0
    and test_pr_auc > 0
)

risk_bands_valid = (
    risk_band_summary.count() > 0
)

model_explanation_valid = (
    explanation_df.count() > 0
)


quality_checks = {
    "primary_target_default_90d": target_valid,

    "temporal_split_used": temporal_split_valid,

    "forbidden_rolling_default_features_absent": (
        len(forbidden_present_final) == 0
    ),

    "full_information_features": len(full_features),

    "behavioral_transaction_features": (
        len(behavioral_model_features)
    ),

    "four_models_compared": four_models_compared,

    "validation_used_for_selection": (
        validation_selection_valid
    ),

    "final_model_frozen": model_selection_valid,

    "out_of_time_test_2024": oot_test_valid,

    "risk_bands_created": risk_bands_valid,

    "model_explanation_created": model_explanation_valid
}


print("============================================================")
print("FINAL MODELLING QUALITY CHECKS")
print("============================================================")

for check_name, result in quality_checks.items():

    print(
        f"{check_name}: {result}"
    )

    if isinstance(result, bool) and not result:
        raise ValueError(
            f"Quality gate failed: {check_name}"
        )


print("============================================================")
print("BNPL CLASSIFICATION QUALITY GATE PASSED")
print("============================================================")

print(
    f"Primary target: {final_target}"
)

print(
    f"Selected model: {final_model_name}"
)

print(
    f"Frozen validation threshold: "
    f"{final_threshold:.3f}"
)

print(
    f"2023 Validation ROC-AUC: "
    f"{float(selected_row['roc_auc']):.4f}"
)

print(
    f"2023 Validation PR-AUC: "
    f"{float(selected_row['pr_auc']):.4f}"
)

print(
    f"2024 OOT ROC-AUC: "
    f"{test_roc_auc:.4f}"
)

print(
    f"2024 OOT PR-AUC: "
    f"{test_pr_auc:.4f}"
)

print(
    "Point-in-time leakage controls: PASSED"
)

print(
    "Temporal validation: PASSED"
)

print(
    "Full vs Behavioural comparison: PASSED"
)

print(
    "Risk deciles and bands: PASSED"
)

print(
    "Model explanation: PASSED"
)

print("============================================================")

FINAL MODELLING QUALITY CHECKS
primary_target_default_90d: True
temporal_split_used: True
forbidden_rolling_default_features_absent: True
full_information_features: 22
behavioral_transaction_features: 20
four_models_compared: True
validation_used_for_selection: True
final_model_frozen: True
out_of_time_test_2024: True
risk_bands_created: True
model_explanation_created: True
BNPL CLASSIFICATION QUALITY GATE PASSED
Primary target: default_90d
Selected model: Random Forest - Full Information
Frozen validation threshold: 0.150
2023 Validation ROC-AUC: 0.9490
2023 Validation PR-AUC: 0.5685
2024 OOT ROC-AUC: 0.9494
2024 OOT PR-AUC: 0.5715
Point-in-time leakage controls: PASSED
Temporal validation: PASSED
Full vs Behavioural comparison: PASSED
Risk deciles and bands: PASSED
Model explanation: PASSED
